# 08 · Derivatives

`s.derivative(method, n=1 or 2, **settings)`. A 1st derivative removes a constant offset and turns each band top into a zero crossing; a 2nd derivative removes a straight-line background and turns each band into a sharp minimum, which separates overlapping bands. Differentiating amplifies noise, so each method smooths at the same time. Units become y per x (per x² for n=2).

| method | settings (None = tuned for n) | note |
|---|---|---|
| `savgol` (default) | `window` 15 / `order` 5 for n=1, 11 / 4 for n=2 | best in the comparison |
| `gaussian` | `sigma` 1.25 for n=1, 1.5 for n=2 | one setting |
| `spline` | `lam` None = automatic | good for n=1, poor for n=2 |
| `gradient` | none | no smoothing; only for smooth data |

How they compare: `docs/benchmarks.md`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import spectrum_toolkit as stk
from spectrum_toolkit.derivative import METHODS, DEFAULTS

cv = stk.load('../data/CV10-4M.xlsx', verbose=False).despike(verbose=False)
s = cv[0]

## 1. Raw spectrum, 1st and 2nd derivative (all methods), zoomed on the 1540-1660 cm⁻¹ bands

In [ ]:
m = (s.x > 1540) & (s.x < 1660)
fig, ax = plt.subplots(3, 1, figsize=(12, 9), sharex=True)
ax[0].plot(s.x[m], s.y[m], 'k'); ax[0].set_title('spectrum (no baseline correction needed)')
for n, a in ((1, ax[1]), (2, ax[2])):
    for meth in METHODS:
        a.plot(s.x[m], s.derivative(meth, n=n).y[m], lw=1, label=meth)
    a.axhline(0, color='0.7', lw=0.5); a.set_title(f'derivative n={n}'); a.legend(fontsize=7)
plt.show()

## 2. Play with the settings

In [ ]:
from ipywidgets import interact, Dropdown, IntSlider, FloatSlider, Checkbox

def show(method='savgol', n=1, auto=True, window=15, order=5, sigma=1.25):
    given = {} if auto else dict(window=window, order=order, sigma=sigma)
    settings = {'n': n, **{k: v for k, v in given.items() if k in DEFAULTS[method]}}
    try:
        d = s.derivative(method, **settings)
    except ValueError as e:
        print(e); return
    fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
    ax[0].plot(s.x, s.y, 'k', lw=0.7)
    ax[1].plot(d.x, d.y, 'b', lw=0.8); ax[1].axhline(0, color='0.7', lw=0.5)
    ax[1].set_title(f"{method}: {d.history[-1]}"); ax[1].set_ylabel(d.y_unit)
    plt.show()

interact(show, method=Dropdown(options=METHODS), n=Dropdown(options=[1, 2]),
         window=IntSlider(value=15, min=5, max=51, step=2), order=IntSlider(value=5, min=1, max=8),
         sigma=FloatSlider(value=1.25, min=0.5, max=6, step=0.25));

Untick `auto` to set the window, order or sigma yourself. A wider window or sigma gives a cleaner but blunter derivative.

## 3. A whole collection, after normalisation

In [ ]:
d2 = cv.normalize().derivative(n=2)
d2.history